# 🥑 Greenpod Labs Assignment 2: Exploratory Data Analysis & Bias Detection

**Objective:** Inspect class distributions, resolution shifts, color distributions (CIELAB & HSV), specular reflections, and potential dataset biases for produce ripening stage and defect classification.


In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='deep')


## 1. Verify Dataset Directory and Integrity
Check that all images can be loaded without corruption and verify formats.


In [ ]:
from src.dataset import prepare_dataset_splits, scan_class_folders

splits = prepare_dataset_splits('.')
classes = splits['classes']
print('Detected Classes:', classes)
print(f"Train set: {len(splits['train'][0])} images")
print(f"Val set:   {len(splits['val'][0])} images")
print(f"Test set:  {len(splits['test'][0])} images")


## 2. Class Distribution & Imbalance Analysis
Evaluate class imbalance across categories to justify class-weighted loss and stratification.


In [ ]:
tr_paths, tr_labels = splits['train']
val_paths, val_labels = splits['val']
te_paths, te_labels = splits['test']

train_counts = [sum(1 for l in tr_labels + val_labels if l == i) for i in range(len(classes))]
test_counts = [sum(1 for l in te_labels if l == i) for i in range(len(classes))]

df_dist = pd.DataFrame({
    'Class': classes,
    'Train+Val Count': train_counts,
    'Test Count': test_counts,
    'Total': [a + b for a, b in zip(train_counts, test_counts)],
    'Percentage (%)': [round((a + b) / sum(train_counts + test_counts) * 100, 2) for a, b in zip(train_counts, test_counts)]
})
print(df_dist)


## 3. Resolution & Image Dimension Shift (Data Bias)
Analyze pixel width and height distributions across classes. Notice that fresh images are smaller PNGs while diseases are 800x800 JPGs!


In [ ]:
records = []
for p, l in zip(tr_paths + te_paths, tr_labels + te_labels):
    with Image.open(p) as img:
        records.append({'class': classes[l], 'width': img.width, 'height': img.height, 'format': img.format})

df_res = pd.DataFrame(records)
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.boxplot(data=df_res, x='class', y='width', hue='class', legend=False)
plt.title('Width Distribution (Resolution Shift Bias)')
plt.subplot(1, 2, 2)
sns.countplot(data=df_res, x='class', hue='format')
plt.title('File Format Distribution (PNG vs JPG)')
plt.tight_layout()
plt.show()


## 4. Color Channel Separation (CIELAB a* & HSV)
Examine how CIELAB a* channel isolates green vs red/orange tones and HSV separates hue angles.


In [ ]:
from src.dataset import clean_load_image

fig, axes = plt.subplots(len(classes), 3, figsize=(11, 2.8 * len(classes)))
for idx, cls_name in enumerate(classes):
    sample_path = [p for p, l in zip(tr_paths, tr_labels) if classes[l] == cls_name][0]
    img = np.array(clean_load_image(sample_path).resize((128, 128)))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    axes[idx, 0].imshow(img); axes[idx, 0].set_title(f"{cls_name} (RGB)"); axes[idx, 0].axis('off')
    axes[idx, 1].imshow(lab[:, :, 1], cmap='PiYG'); axes[idx, 1].set_title('CIELAB a* (Green-Red)'); axes[idx, 1].axis('off')
    axes[idx, 2].imshow(hsv[:, :, 0], cmap='hsv'); axes[idx, 2].set_title('HSV Hue Angle'); axes[idx, 2].axis('off')
plt.tight_layout()
plt.show()
